# In Class Assignment 6
Building a Gaussian Naive Bayes model to predict irrigation need using the Kaggle S6E4 competition dataset.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, balanced_accuracy_score

In [2]:
# Load the Kaggle train data, inspect it, and prepare the target variable
# Map target to numeric classes: Low (0), Medium (1), High (2)
df = pd.read_csv('../data/train.csv')

print(df.head())
print("\nTarget Distribution:")
print(df['Irrigation_Need'].value_counts())

df['Irrigation_Need'] = df['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})

# We exclude 'id' and 'Irrigation_Need' strings from X
X = df.drop(columns=['id', 'Irrigation_Need'])
y = df['Irrigation_Need']

# Train/Test Split (only using Kaggle train data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [4]:
# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Define preprocessing pipeline
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Helper for converting sparse matrices to dense arrays for Naive Bayes
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

# Define full Naive Bayes model pipeline
nb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", FunctionTransformer(to_dense, accept_sparse=True)),
    ("model", GaussianNB())
])

# Train the model on the Kaggle train split
nb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Soil_Type', 'Crop_Type',
                                                   'Crop_Growth_Stage',
                                                   'Season', 'Irrigation_Type',
                                                   'Water_Source',
                                                   'Mulching_Used',
                                                   'Region'])])),
                ('to_dense',
                 FunctionTransformer(accept_sparse=True,
                                     func=<function to_dense at 0x163306340>)),
                ('model', GaussianNB())])

In [5]:
# Generate probabilities and default baseline predictions
probs = nb_model.predict_proba(X_test)

# Baseline prediction (default rule: highest probability class)
preds_baseline = probs.argmax(axis=1)

# Evaluate Baseline Model
print("--- Baseline Evaluation ---")
print("Baseline Recall for High (Class 2):", recall_score(y_test, preds_baseline, average=None)[2])
print("\nBaseline Classification Report:")
print(classification_report(y_test, preds_baseline, target_names=["Low", "Medium", "High"]))


--- Baseline Evaluation ---
Baseline Recall for High (Class 2): 0.6261304140885293

Baseline Classification Report:
              precision    recall  f1-score   support

         Low       0.86      0.73      0.79     73983
      Medium       0.57      0.50      0.54     47815
        High       0.13      0.63      0.21      4202

    accuracy                           0.64    126000
   macro avg       0.52      0.62      0.51    126000
weighted avg       0.73      0.64      0.67    126000



In [6]:
# --- Threshold Tuning ---
# Choose class to focus on: "High" (mapped to 2)
# Since High irrigation means water is needed, we want a high recall 
# to ensure we don't miss fields that require significant irrigation.
target_class = 2   
threshold = 0.15   # Lower threshold to prioritize identifying 'High' need

# Apply threshold
preds_threshold = preds_baseline.copy()
mask = probs[:, target_class] >= threshold
preds_threshold[mask] = target_class

# Evaluate Threshold Model
print(f"--- Threshold Evaluation (Class High >= {threshold}) ---")
print("Threshold Recall for High (Class 2):", recall_score(y_test, preds_threshold, average=None)[2])
print("\nThreshold Classification Report:")
print(classification_report(y_test, preds_threshold, target_names=["Low", "Medium", "High"]))


--- Threshold Evaluation (Class High >= 0.15) ---
Threshold Recall for High (Class 2): 0.8457877201332699

Threshold Classification Report:
              precision    recall  f1-score   support

         Low       0.86      0.73      0.79     73983
      Medium       0.50      0.32      0.39     47815
        High       0.11      0.85      0.20      4202

    accuracy                           0.58    126000
   macro avg       0.49      0.63      0.46    126000
weighted avg       0.70      0.58      0.62    126000



### Discussion

* **Chosen Class:** We selected **"High"** (class 2) because it represents fields that require the most water. In agricultural contexts, failing to irrigate a field that has a "High" need (a false negative) could lead to significant crop loss. Therefore, identifying as many true "High" cases as possible is a priority.
* **Evaluation Metric:** **Recall** for the "High" class. We want to minimize false negatives for the "High" class.
* **Chosen Threshold:** We lowered the threshold for predicting "High" from the default logic (simply taking argmax among 3 classes) down to an explicit **0.15** probability threshold.
* **Observed Tradeoffs:** By lowering the threshold, the recall for the "High" class increased significantly (meaning we successfully identified more fields requiring high irrigation). However, this came at the expense of precision and F1-score for the "High" class, as well as a slight drop in overall accuracy. More fields that actually required "Low" or "Medium" irrigation were incorrectly flagged as "High" (false positives).
* **Comparison to Existing Models:** Compared to more complex models (like XGBoost or Random Forest seen in previous assignments), Naive Bayes is significantly faster to train and uses less memory, particularly when utilizing a dense transformation step. However, its overall predictive performance (accuracy, F1) is somewhat lower due to its "naive" assumption of conditional independence between features like soil, temperature, and region. Despite this, the thresholding technique allows us to manually tune its utility towards our high-risk class (High irrigation need), making it a highly competitive and interpretable baseline.